# LLM-based classification of the literal reason for the emergency call — two-axis taxonomy

Supplementary material for: *<paper title>*.

**Pipeline.** The literal reason for the emergency call (free text recorded by the call-taker; one entry per emergency demand) is restricted to the study cohort, normalised and classified by a locally served large language model (LLM) into a two-axis, multi-label taxonomy: **Axis 1 – demand** (61 labels, `ml_dem_*`: what the call is about) and **Axis 2 – findings** (85 labels: signs and symptoms, time course, frailty, history, objective vital signs and caller context). The two axes are independent and are encoded as two separate one-hot blocks per demand.

**Availability semantics** (per demand, column `estado`): `OK` = at least one label; `SIN_HALLAZGO` = text was classified but no label applied; `SIN_TEXTO` = no usable text (missing, empty or an empty-equivalent token such as "idem"); `NO_CLASIFICADO` = text available but the LLM request failed. Each Axis-1 label maps one-to-one to a category of the dispatch centre's demand-type catalogue (A.00–A.60); the mapping is informational and is not used for evaluation.

**Data availability.** The source data contain identifiable health information and cannot be shared. <data availability statement>. This notebook documents the method; file locations are placeholders and all cell outputs have been removed.

**Computational environment.** Ollama <version>, model `<model tag>` (digest `<digest>`), single GPU with 24 GB VRAM, Python <version>, pandas <version>, NumPy <version>.

## 1. Configuration

In [ ]:
import hashlib
import json
import random
import re
import time
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import ollama
import pandas as pd

# Input/output locations (placeholders: the source data are not publicly available)
DATA_DIR = Path('data')
OUT_DIR = Path('output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'cohort':  DATA_DIR / 'cohort.parquet',   # demand_id
    'reasons': DATA_DIR / 'reasons.parquet',  # demand_id, text
}

# LLM configuration: exactly the options sent to the model in the reported run
MODEL = 'qwen2.5:32b-instruct-q3_K_M'
LLM_OPTIONS = {'temperature': 0.0, 'num_ctx': 12288, 'num_predict': 256}
KEEP_ALIVE = '10m'
N_WORKERS = 1    # concurrent requests (each parallel slot allocates its own num_ctx KV cache)
RETRIES = 2      # retries on transport errors only
CHUNK = 10_000   # texts submitted per batch (bounds memory and flushes the cache regularly)
SEED = 42

## 2. Input data

Two tables are required. Identifiers are pseudonymised; free text is used as recorded (no spelling correction).

| Table | Grain | Columns |
|---|---|---|
| `cohort` | one row per emergency demand of the study cohort | `demand_id` |
| `reasons` | one row per demand (deduplicated beforehand) | `demand_id`, `text` (literal reason for the call) |

Cohort demands without an entry in `reasons`, or with an empty entry, are retained and labelled `SIN_TEXTO`. <Describe how demands with more than one source record were deduplicated upstream.> No additional filtering is applied.

In [ ]:
KEY_DEM = 'demand_id'
cohort = pd.read_parquet(PATHS['cohort'], columns=[KEY_DEM])
reasons = pd.read_parquet(PATHS['reasons'], columns=[KEY_DEM, 'text'])
assert reasons[KEY_DEM].is_unique

demands = (cohort.dropna(subset=[KEY_DEM]).drop_duplicates(KEY_DEM)
           .merge(reasons, on=KEY_DEM, how='left')
           .reset_index(drop=True))
assert demands[KEY_DEM].is_unique
print(f'Cohort demands: {len(demands):,} | with non-null text: {demands["text"].notna().sum():,}')

## 3. Taxonomy and prompt

Axis 1 contains 61 demand labels, each mapped one-to-one to the dispatch centre's demand-type catalogue (A.00–A.60). Axis 2 contains 85 findings grouped by clinical domain (stroke focal deficits, chest pain, dyspnoea, consciousness, haemorrhage, trauma mechanism, burns, obstetric, suicidal behaviour) plus transversal blocks (shared findings, time course, frailty, history, toxics, objective vital signs, logistics). <Describe how the taxonomy was developed and reviewed by clinicians, and who set the vital-sign thresholds.> The prompt is reproduced verbatim. The model output is constrained by a JSON schema to an array of unique taxonomy labels, which is the output format requested in the prompt.

In [ ]:
# Axis-1 label -> code of the dispatch centre's demand-type catalogue (informational only)
DEMAND_TO_OPERATOR_CODE = {
    'ml_dem_autoactiv':             0,  # A.00 AUTOACTIVACIÓN
    'ml_dem_toxicos':               1,  # A.01 ABUSO DE TÓXICOS, DROGAS, MEDICAMENTOS Y/O ALCOHOL
    'ml_dem_buceo':                 2,  # A.02 ACCIDENTE DE BUCEO
    'ml_dem_accid_transp':          3,  # A.03 ACCIDENTE DE TRANSPORTE
    'ml_dem_accid_electr':          4,  # A.04 ACCIDENTE ELÉCTRICO
    'ml_dem_agitacion':             5,  # A.05 AGITACIÓN Y/O VIOLENCIA
    'ml_dem_miccion':               6,  # A.06 ALTERACIONES DE LA MICCIÓN
    'ml_dem_amenaza_aborto':        7,  # A.07 AMENAZA DE ABORTO
    'ml_dem_suicida':               8,  # A.08 CONDUCTA SUICIDA
    'ml_dem_aplast_amput':          9,  # A.09 APLASTAMIENTO Y/O AMPUTACIÓN
    'ml_dem_caida':                10,  # A.10 CAIDA/TROPIEZO
    'ml_dem_cefalea':              11,  # A.11 CEFALEAS
    'ml_dem_contusion':            12,  # A.12 CONTUSIÓN
    'ml_dem_convulsion':           13,  # A.13 CONVULSIÓN
    'ml_dem_enfermeria':           14,  # A.14 DEMANDAS DE ENFERMERÍA
    'ml_dem_depresion':            15,  # A.15 DEPRESIÓN/TRISTEZA
    'ml_dem_disnea':               16,  # A.16 DIFICULTAD RESPIRATORIA
    'ml_dem_dolor_abdo':           17,  # A.17 DOLOR ABDOMINAL
    'ml_dem_dolor_cervical':       18,  # A.18 DOLOR CERVICAL
    'ml_dem_dolor_espalda':        19,  # A.19 DOLOR DE ESPALDA/COLUMNA
    'ml_dem_dolor_extrem':         20,  # A.20 DOLOR DE EXTREMIDADES/ARTICULAR
    'ml_dem_dolor_generaliz':      21,  # A.21 DOLOR GENERALIZADO/MAL DEFINIDO
    'ml_dem_dolor_genitourin':     22,  # A.22 DOLOR GÉNITOURINARIO/PÉLVICO
    'ml_dem_toracico':             23,  # A.23 DOLOR TORÁCICO
    'ml_dem_estren_diarrea':       24,  # A.24 ESTREÑIMIENTO/DIARREAS
    'ml_dem_fiebre':               25,  # A.25 FIEBRE
    'ml_dem_hemorr_gineco':        26,  # A.26 HEMORRAGIA GINECOLÓGICA
    'ml_dem_hemorr_anal':          27,  # A.27 HEMORRAGIA POR VÍA ANAL Y/O HECES NEGRAS
    'ml_dem_hemorr_oral':          28,  # A.28 HEMORRAGIA POR VÍA ORAL
    'ml_dem_hemorr_urinaria':      29,  # A.29 HEMORRAGIA POR VÍA URINARIA
    'ml_dem_arma_blanca_fuego':    30,  # A.30 HERIDA POR ARMA BLANCA/FUEGO
    'ml_dem_heridas_hemorr':       31,  # A.31 HERIDAS/HEMORRAGIAS
    'ml_dem_hiperglucemia':        32,  # A.32 HIPERGLUCEMIA
    'ml_dem_hipertension':         33,  # A.33 HIPERTENSIÓN
    'ml_dem_hipoglucemia':         34,  # A.34 HIPOGLUCEMIA
    'ml_dem_hipotension':          35,  # A.35 HIPOTENSIÓN
    'ml_dem_inconsciente':         36,  # A.36 INCONSCIENTE
    'ml_dem_malestar_gral':        37,  # A.37 MALESTAR GENERAL
    'ml_dem_mareo_sincope':        38,  # A.38 MAREOS/DESVANECIMIENTOS/SÍNCOPE
    'ml_dem_mordedura_picadura':   39,  # A.39 MORDEDURAS/PICADURAS
    'ml_dem_nauseas_vomitos':      40,  # A.40 NAUSEAS Y VÓMITOS
    'ml_dem_ansiedad':             41,  # A.41 NERVIOSISMO/ANSIEDAD
    'ml_dem_obstr_via_aerea':      42,  # A.42 OBSTRUCCIÓN VÍA AÉREA POR CUERPO EXTRAÑO
    'ml_dem_otras_alt_conciencia': 43,  # A.43 OTRAS ALTERACIONES DEL NIVEL DE CONCIENCIA
    'ml_dem_otros_valdificil':     44,  # A.44 OTROS MOTIVOS/VALORACIÓN DIFÍCIL
    'ml_dem_palpitaciones':        45,  # A.45 PALPITACIONES/ARRITMIAS
    'ml_dem_parto':                46,  # A.46 PARTO Y ALUMBRAMIENTO DE URGENCIAS
    'ml_dem_precipitacion':        47,  # A.47 PRECIPITACIÓN
    'ml_dem_orl':                  48,  # A.48 PROBLEMAS BOCA/GARGANTA/NARIZ Y OÍDOS
    'ml_dem_dermatologico':        49,  # A.49 PROBLEMAS DERMATOLÓGICOS
    'ml_dem_paliativos':           50,  # A.50 PROBLEMAS EN PACIENTES DE CUIDADOS PALIATIVOS
    'ml_dem_oftalmologico':        51,  # A.51 PROBLEMAS OFTALMOLÓGICOS
    'ml_dem_quemaduras':           52,  # A.52 QUEMADURAS
    'ml_dem_semiahogamiento':      53,  # A.53 SEMIAHOGAMIENTO
    'ml_dem_ictus':                54,  # A.54 SOSPECHA ACV
    'ml_dem_gripe':                55,  # A.55 SOSPECHA DE GRIPE
    'ml_dem_malos_tratos':         56,  # A.56 SOSPECHA DE MALOS TRATOS
    'ml_dem_exitus':               57,  # A.57 SOSPECHA EXITUS
    'ml_dem_pcr':                  58,  # A.58 SOSPECHA PARADA CARDIORRESPIRATORIA
    'ml_dem_covid':                59,  # A.59 SÍNTOMAS COMPATIBLES COVID-19
    'ml_dem_cronico_complejo':     60,  # A.60 VALORACIÓN Y ATENCIÓN DE PACIENTES CRÓNICOS COMPLEJOS
}

# Full vocabulary in stable order (defines the one-hot columns and the output enum)
ML_COLUMNS = [
    # ===== Axis 1: demand (multi-label) =====
    'ml_dem_ictus', 'ml_dem_toracico', 'ml_dem_disnea', 'ml_dem_inconsciente',
    # neurological / consciousness / syncope
    'ml_dem_convulsion', 'ml_dem_otras_alt_conciencia', 'ml_dem_mareo_sincope', 'ml_dem_cefalea',
    # cardiorespiratory / vital-sign demands
    'ml_dem_palpitaciones', 'ml_dem_pcr', 'ml_dem_obstr_via_aerea', 'ml_dem_semiahogamiento',
    'ml_dem_hipertension', 'ml_dem_hipotension', 'ml_dem_hiperglucemia', 'ml_dem_hipoglucemia',
    # pain
    'ml_dem_dolor_abdo', 'ml_dem_dolor_cervical', 'ml_dem_dolor_espalda', 'ml_dem_dolor_extrem',
    'ml_dem_dolor_generaliz', 'ml_dem_dolor_genitourin',
    # haemorrhage and wounds
    'ml_dem_hemorr_gineco', 'ml_dem_hemorr_anal', 'ml_dem_hemorr_oral', 'ml_dem_hemorr_urinaria',
    'ml_dem_heridas_hemorr', 'ml_dem_arma_blanca_fuego',
    # trauma / accidents
    'ml_dem_caida', 'ml_dem_contusion', 'ml_dem_aplast_amput', 'ml_dem_accid_transp',
    'ml_dem_accid_electr', 'ml_dem_precipitacion', 'ml_dem_buceo', 'ml_dem_quemaduras',
    'ml_dem_mordedura_picadura',
    # digestive / genitourinary
    'ml_dem_nauseas_vomitos', 'ml_dem_estren_diarrea', 'ml_dem_miccion',
    # infectious / general
    'ml_dem_fiebre', 'ml_dem_gripe', 'ml_dem_covid', 'ml_dem_malestar_gral',
    # obstetric
    'ml_dem_amenaza_aborto', 'ml_dem_parto',
    # mental health / behaviour
    'ml_dem_agitacion', 'ml_dem_suicida', 'ml_dem_depresion', 'ml_dem_ansiedad',
    'ml_dem_malos_tratos', 'ml_dem_toxicos',
    # focal specialties
    'ml_dem_orl', 'ml_dem_dermatologico', 'ml_dem_oftalmologico',
    # administrative / low clinical content
    'ml_dem_exitus', 'ml_dem_paliativos', 'ml_dem_cronico_complejo', 'ml_dem_enfermeria',
    'ml_dem_otros_valdificil', 'ml_dem_autoactiv',

    # ===== Axis 2: findings =====
    # stroke: focal deficits
    'ml_foc_habla', 'ml_foc_facial', 'ml_foc_motor_ms', 'ml_foc_lateral',
    'ml_foc_motor_mi', 'ml_foc_sensiti', 'ml_foc_visual', 'ml_foc_equilibrio', 'ml_cefalea_max',
    # chest pain
    'ml_tor_opresivo', 'ml_tor_pleuritico', 'ml_tor_irrad_tipica',
    'ml_tor_irrad_dorsal', 'ml_tor_irrad_epigast', 'ml_tor_esfuerzo', 'ml_tor_reposo',
    # dyspnoea
    'ml_dis_trabajo', 'ml_dis_ortopnea', 'ml_dis_sibilancias', 'ml_dis_cianosis',
    'ml_dis_edemas', 'ml_dis_expector', 'ml_dis_hemoptisis', 'ml_dis_cuerpo_extrano',
    'ml_dis_tep_riesgo',
    # consciousness
    'ml_inc_no_respira', 'ml_inc_no_responde', 'ml_inc_transitorio', 'ml_inc_convul',
    'ml_inc_hipoglucemia', 'ml_inc_pulso_no', 'ml_confu_aguda', 'ml_trauma_previo',
    # haemorrhage
    'ml_hem_cuantiosa', 'ml_hem_melenas', 'ml_hem_rectorragia', 'ml_hem_hematemesis',
    'ml_hem_epistaxis', 'ml_hem_hematuria', 'ml_hem_vaginal',
    # trauma mechanism
    'ml_trau_alta_energia', 'ml_trau_tce', 'ml_trau_amputacion', 'ml_trau_hemorragia_ext',
    # burns
    'ml_qem_via_aerea', 'ml_qem_extensa', 'ml_qem_electrica_quimica',
    # obstetric
    'ml_obs_parto_inminente', 'ml_obs_sangrado_gestante',
    # suicidal behaviour
    'ml_sui_gesto_actual', 'ml_sui_ideacion',
    # shared findings
    'ml_veg_sudor', 'ml_veg_digest', 'ml_veg_mareo', 'ml_dolor_cabeza',
    'ml_temblor', 'ml_disfagia', 'ml_fiebre', 'ml_palpitaciones',
    'ml_dis_anafilaxia', 'ml_cuadro_catarral',
    # time course
    'ml_tmp_hiperag', 'ml_tmp_despert', 'ml_tmp_evolucionado',
    # frailty
    'ml_frag_social', 'ml_frag_basal',
    # history
    'ml_ant_cardio', 'ml_ant_acv_prev', 'ml_ant_epoc_asma', 'ml_riesgo_cv',
    'ml_ant_epilepsia', 'ml_ant_tep_prev', 'ml_ant_oncologico', 'ml_oxigenoterapia',
    # toxics
    'ml_tox_consumo',
    # objective vital signs (thresholds in prompt rule 21)
    'ml_vit_hipoxia', 'ml_vit_hipoglucemia', 'ml_vit_hiperglucemia',
    'ml_vit_hipertension', 'ml_vit_hipotension', 'ml_vit_bradicardia',
    'ml_vit_taquicardia', 'ml_vit_alterada_referida',
    # logistics
    'ml_src_profes', 'ml_missing_dat',
]
ML_COLUMNS_SET = set(ML_COLUMNS)
DEM_COLS = [c for c in ML_COLUMNS if c.startswith('ml_dem_')]
FIND_COLS = [c for c in ML_COLUMNS if not c.startswith('ml_dem_')]

assert len(ML_COLUMNS) == len(ML_COLUMNS_SET) == 146
assert len(DEM_COLS) == 61 and len(FIND_COLS) == 85
assert set(DEM_COLS) == set(DEMAND_TO_OPERATOR_CODE)
assert sorted(DEMAND_TO_OPERATOR_CODE.values()) == list(range(61))

In [ ]:
SYSTEM_PROMPT = """\
You are an expert clinical NLP system specialized in prehospital emergency triage of Spanish emergency call transcripts (061 / SEM).
Your ONLY task: assign all applicable category labels to the input text (the free-text field MOTIVOLITERAL).

The taxonomy has TWO INDEPENDENT AXES that may co-occur:
  - AXIS 1 (DEMANDA): what the call is ABOUT. MULTI-LABEL: a single call may
    belong to several demands at once (e.g. chest pain WITH dyspnea). Each
    demand label maps 1:1 to the official operator category (IDTIPODEMANDA1).
  - AXIS 2 (HALLAZGOS): concrete signs/symptoms described, regardless of demand.

GENERAL PRINCIPLE FOR AXIS 1: activate a demand label whenever the literal text
describes that clinical problem as a reason for the call, even on suspicion
("posible ictus", "parece un cólico"). Prefer the MOST SPECIFIC demand that the
text supports. Do NOT invent a demand that is not described. If several specific
demands are described, activate all of them (multi-label).

TAXONOMY (Label | Definition | Spanish trigger terms & synonyms):

=== AXIS 1 — DEMANDA (multi-label; each maps 1:1 to IDTIPODEMANDA1) ===
  ml_dem_ictus        | A.54 Suspected stroke/TIA | ictus, ACV, ACVA, AIT, infarto cerebral, isquemia cerebral, derrame, código ictus, embolia cerebral, se le ha torcido la cara, no habla de repente
  ml_dem_toracico     | A.23 Chest pain/discomfort | dolor de pecho, dolor torácico, opresión en el pecho, angina, me duele el pecho, dolor precordial, presión en el pecho
  ml_dem_disnea       | A.16 Breathing difficulty | disnea, dificultad para respirar, no puede respirar, se ahoga, falta de aire, ahogo, se asfixia, le falta el aire, respira mal, fatiga respiratoria
  ml_dem_inconsciente | A.36 Loss/alteration of consciousness as the demand | inconsciente, no responde, no reacciona, desmayado, desplomado, pérdida de conocimiento, no despierta, no se le despierta, desvanecimiento
  ml_dem_convulsion   | A.13 Seizure as the demand | convulsión, convulsiona, ataque epiléptico, crisis convulsiva, espasmos con pérdida de conocimiento, está convulsionando
  ml_dem_otras_alt_conciencia | A.43 Other altered consciousness (obtundation/stupor/somnolence not fitting inconsciente/síncope) | muy adormilado, no se le despierta bien, estuporoso, obnubilado, bajo nivel de conciencia, responde poco, muy decaído sin responder bien
  ml_dem_mareo_sincope| A.38 Dizziness / faint / syncope as the demand | mareo, mareado, vértigo, desvanecimiento, síncope, lipotimia, se ha desmayado y se ha recuperado, sensación de desmayo
  ml_dem_cefalea      | A.11 Headache as the demand | dolor de cabeza, cefalea, jaqueca, migraña, mucho dolor de cabeza, le duele mucho la cabeza
  ml_dem_palpitaciones| A.45 Palpitations / arrhythmia as the demand | palpitaciones, arritmia, el corazón le va muy rápido, taquicardia, fibrilación, el corazón se le sale, vuelcos en el corazón, se le acelera el corazón
  ml_dem_pcr          | A.58 Suspected cardiac arrest as the demand | parada, no respira y no responde, parada cardiaca, parada cardiorrespiratoria, PCR, hay que hacer reanimación, no tiene pulso ni respira
  ml_dem_obstr_via_aerea | A.42 Foreign-body airway obstruction as the demand | se ha atragantado, atragantamiento, se ahoga con la comida, cuerpo extraño en la garganta, no puede respirar por algo atascado, maniobra de Heimlich
  ml_dem_semiahogamiento | A.53 Near-drowning / submersion | se ha ahogado en el agua, semiahogamiento, casi se ahoga en la piscina, lo han sacado del agua, sumersión, ahogamiento en el mar
  ml_dem_hipertension | A.33 Hypertension as the demand | tensión alta, hipertensión, la tensión disparada, tensión muy alta, crisis hipertensiva, subida de tensión
  ml_dem_hipotension  | A.35 Hypotension as the demand | tensión baja, hipotensión, la tensión por los suelos, se le ha bajado la tensión, tensión muy baja
  ml_dem_hiperglucemia| A.32 Hyperglycemia as the demand | azúcar alto, glucemia alta, hiperglucemia, el azúcar disparado, diabético con azúcar muy alto
  ml_dem_hipoglucemia | A.34 Hypoglycemia as the demand | bajada de azúcar, azúcar baja, hipoglucemia, se le ha bajado el azúcar, diabético con el azúcar bajo
  ml_dem_dolor_abdo   | A.17 Abdominal pain | dolor de barriga, dolor abdominal, dolor de tripa, retortijones, cólico, dolor de estómago, dolor en el vientre, dolor de vientre
  ml_dem_dolor_cervical | A.18 Neck pain | dolor de cuello, dolor cervical, cervicalgia, tortícolis, dolor en la nuca
  ml_dem_dolor_espalda| A.19 Back/spine pain | dolor de espalda, lumbago, lumbalgia, dolor de riñones, dolor de columna, dolor dorsal, ciática
  ml_dem_dolor_extrem | A.20 Limb/joint pain | dolor en el brazo, dolor en la pierna, dolor articular, dolor de rodilla, dolor de hombro, dolor en una extremidad, no puede mover por dolor
  ml_dem_dolor_generaliz | A.21 Generalized/ill-defined pain | dolor por todo el cuerpo, dolor generalizado, le duele todo, dolores difusos, malestar con dolor mal definido
  ml_dem_dolor_genitourin | A.22 Genitourinary/pelvic pain | dolor al orinar, dolor genital, dolor testicular, dolor pélvico, cólico nefrítico, dolor en la zona íntima, dolor de ovarios
  ml_dem_hemorr_gineco| A.26 Gynecological bleeding | sangrado vaginal, hemorragia ginecológica, sangra por la vagina, regla muy abundante, metrorragia (NOT pregnant — if pregnant see A.07/A.46)
  ml_dem_hemorr_anal  | A.27 Anal bleeding / black stools | sangre por el ano, sangre al defecar, heces negras, melenas, rectorragia, sangra por detrás
  ml_dem_hemorr_oral  | A.28 Oral bleeding (mouth/nose/vomit) | sangre por la boca, vómito con sangre, hematemesis, sangra por la nariz, epistaxis, echa sangre por la boca
  ml_dem_hemorr_urinaria | A.29 Urinary bleeding | sangre en la orina, hematuria, orina con sangre, orina roja
  ml_dem_heridas_hemorr | A.31 Wounds / hemorrhage (general, traumatic or unspecified) | herida que sangra, corte que sangra, hemorragia, sangra mucho de una herida, herida abierta
  ml_dem_arma_blanca_fuego | A.30 Stab / gunshot wound | apuñalado, herida de arma blanca, cuchillada, disparo, herida de bala, arma de fuego, le han pegado un tiro, navajazo
  ml_dem_caida        | A.10 Fall / trip | se ha caído, caída, se cayó, tropezó y cayó, caída desde su altura, se ha caído al suelo
  ml_dem_contusion    | A.12 Contusion / blunt trauma (non-penetrating) | golpe, contusión, moratón, se ha dado un golpe, magulladura, traumatismo cerrado
  ml_dem_aplast_amput | A.09 Crush / amputation | aplastamiento, atrapado, le ha cogido la máquina, amputación, le ha cortado el dedo/mano, miembro amputado, atrapado por peso
  ml_dem_accid_transp | A.03 Traffic/transport accident | accidente de tráfico, accidente de coche, atropello, colisión, accidente de moto, choque, salida de vía, accidente de circulación
  ml_dem_accid_electr | A.04 Electrical accident | descarga eléctrica, electrocución, calambrazo fuerte, accidente eléctrico, contacto con la corriente, le ha dado la corriente
  ml_dem_precipitacion| A.47 Fall from height / jump | precipitación, se ha tirado desde, caída desde altura, se ha caído de un tejado/ventana/balcón, se ha precipitado
  ml_dem_buceo        | A.02 Diving accident | accidente de buceo, descompresión, buceando, submarinismo, ha subido muy rápido buceando
  ml_dem_quemaduras   | A.52 Burns | quemadura, se ha quemado, escaldadura, quemado con, ampollas por quemadura, fuego le ha alcanzado
  ml_dem_mordedura_picadura | A.39 Bites / stings | mordedura, le ha mordido un perro, picadura, le ha picado una abeja/avispa, mordedura de serpiente, picotazo
  ml_dem_nauseas_vomitos | A.40 Nausea / vomiting as the demand | vómitos, náuseas, está vomitando, no para de vomitar, arcadas, devuelve todo
  ml_dem_estren_diarrea | A.24 Constipation / diarrhea | diarrea, descomposición, no hace de vientre, estreñimiento, deposiciones líquidas, cagalera, lleva días sin defecar
  ml_dem_miccion      | A.06 Micturition alterations | no puede orinar, retención de orina, no hace pis, escapes de orina, no expulsa la orina, sonda que no funciona
  ml_dem_fiebre       | A.25 Fever as the demand | fiebre, tiene fiebre, está ardiendo, décimas, febrícula, mucha fiebre, temperatura alta
  ml_dem_gripe        | A.55 Suspected flu | gripe, cuadro gripal, síntomas de gripe, tiene la gripe, gripe con fiebre y dolores
  ml_dem_covid        | A.59 COVID-19 compatible symptoms | covid, coronavirus, síntomas de covid, positivo en covid, contacto covid con síntomas
  ml_dem_malestar_gral| A.37 General malaise (non-specific) | malestar general, se encuentra mal, decaído, no se encuentra bien, flojera, malestar sin especificar, encontrarse mal en general
  ml_dem_amenaza_aborto | A.07 Threatened abortion | amenaza de aborto, embarazada sangra, pérdidas estando embarazada, dolor y sangrado en el embarazo, aborto en curso
  ml_dem_parto        | A.46 Imminent labor / delivery | está de parto, va a dar a luz, contracciones seguidas, ha roto aguas, se ve la cabeza del bebé, parto inminente, alumbramiento
  ml_dem_agitacion    | A.05 Agitation / violence | agitado, agresivo, violento, descontrolado, brote, no se le puede sujetar, agitación psicomotriz, está fuera de sí
  ml_dem_suicida      | A.08 Suicidal behavior | intento de suicidio, se quiere matar, se ha cortado las venas, se ha tomado todas las pastillas para quitarse la vida, autolisis, se quiere tirar
  ml_dem_depresion    | A.15 Depression / sadness | depresión, muy triste, decaído anímicamente, llora sin parar, bajón anímico, no quiere vivir sin plan concreto
  ml_dem_ansiedad     | A.41 Nervousness / anxiety | ansiedad, ataque de ansiedad, nervioso, crisis de ansiedad, angustia, hiperventila por nervios, ataque de pánico
  ml_dem_malos_tratos | A.56 Suspected abuse | malos tratos, maltrato, violencia de género, le han pegado, agresión doméstica, sospecha de abuso
  ml_dem_toxicos      | A.01 Substance/drug/alcohol abuse as the demand | borracho, intoxicación etílica, drogado, sobredosis, ha consumido drogas, abuso de alcohol, intoxicación por drogas
  ml_dem_orl          | A.48 Mouth/throat/nose/ear problems | dolor de oído, dolor de garganta, otitis, no oye, cuerpo extraño en el oído/nariz, problema de garganta, afonía
  ml_dem_dermatologico| A.49 Dermatological problems | erupción, sarpullido, ronchas, urticaria, lesión en la piel, granos, picor en la piel, problema dermatológico
  ml_dem_oftalmologico| A.51 Ophthalmological problems | problema en el ojo, dolor de ojo, ha perdido visión por algo en el ojo, cuerpo extraño en el ojo, ojo rojo, molestia ocular
  ml_dem_exitus       | A.57 Suspected death | ha fallecido, está muerto, sin signos de vida hace tiempo, cadáver, exitus, lleva horas fallecido, rigidez cadavérica
  ml_dem_paliativos   | A.50 Palliative-care patient problem | paciente paliativo, cuidados paliativos, enfermo terminal, control de síntomas en paliativos, paciente en situación terminal
  ml_dem_cronico_complejo | A.60 Complex chronic patient assessment | paciente crónico complejo, pluripatológico, reagudización de crónico, seguimiento de crónico complejo
  ml_dem_enfermeria   | A.14 Nursing demand | cura de herida, cambio de sonda, administración de medicación, cuidados de enfermería, sondaje, curas programadas
  ml_dem_otros_valdificil | A.44 Other / hard-to-assess | otros motivos, valoración difícil, no está claro el motivo, motivo indeterminado, no se puede precisar la demanda
  ml_dem_autoactiv    | A.00 Self-activation (unit self-dispatch, administrative) | autoactivación, la unidad se activa sola, activación propia del recurso

=== AXIS 2 — ICTUS: focal deficits (unchanged from validated stroke prompt) ===
  ml_foc_habla       | Speech/language impairment (NOT confusion, NOT breathlessness — see rule 16) | no habla, se le traba la lengua, disartria, afasia, no articula, habla estropajosa, habla trapajosa, le cuesta hablar, no se le entiende
  ml_foc_facial      | Facial/oral-motor weakness (mouth & face, NOT eyes) | boca torcida, boca ladeada, comisura desviada, cara paralizada, parálisis facial, se le cae la baba
  ml_foc_motor_ms    | Upper-limb weakness/paralysis | brazo sin fuerza, no mueve el brazo, mano sin fuerza, se le caen las cosas, brazo caído, mano dormida
  ml_foc_lateral     | Explicit laterality of deficit | izquierdo, derecho, izq, dcho, lado izquierdo/derecho, medio cuerpo, hemicuerpo, hemiparesia, hemiplejia, izda, dcha
  ml_foc_motor_mi    | Lower-limb weakness/paralysis (explicit leg) | no anda, falla la pierna, pierna sin fuerza, arrastra el pie, pierna paralizada, no le responde la pierna
  ml_foc_sensiti     | Sensory deficit / paresthesia (NOT visual) | hormigueo, acorchamiento, no siente, parte dormida, entumecido, parestesia, lado dormido
  ml_foc_visual      | Visual/oculomotor deficit in a CONSCIOUS patient | no ve bien, vista perdida, pierde visión, ve doble, visión borrosa, se le cae el párpado, un ojo no lo mueve
  ml_foc_equilibrio  | Loss of balance/instability in a conscious patient (lay ataxia, "Balance" of BE-FAST) | no se tiene en pie, se le va para un lado, camina como borracho, se tambalea al andar, pierde el equilibrio, se cae hacia un lado
  ml_cefalea_max     | # ⚠ REVISAR — Thunderclap / worst-of-life headache (HSA suspicion) | el peor dolor de cabeza de su vida, un dolor de cabeza como nunca ha tenido, le explotó la cabeza de repente, dolor de cabeza brutal de golpe

=== AXIS 2 — DOLOR TORÁCICO (qualifiers of ml_dem_toracico) ===
  ml_tor_opresivo      | Oppressive / pressure-like | opresión, como un peso, me aprieta, presión en el pecho, dolor opresivo, como si me apretaran
  ml_tor_pleuritico    | Pleuritic (worse with breathing) | duele al respirar, punzada al coger aire, dolor al inspirar, pinchazo al respirar
  ml_tor_irrad_tipica  | Radiation to left arm / jaw / neck | irradia al brazo, dolor en el brazo izquierdo, dolor en la mandíbula, sube al cuello, se corre al brazo
  ml_tor_irrad_dorsal  | # ⚠ REVISAR — Radiation to back/interscapular (dissection flag) | dolor que va a la espalda, entre los omóplatos, dolor desgarrante a la espalda, dolor interescapular
  ml_tor_irrad_epigast | Epigastric/abdominal radiation (atypical anginal equivalent) | dolor que se va a la barriga, dolor de pecho y estómago, se le va al vientre, dolor desde el estómago hacia arriba
  ml_tor_esfuerzo      | Onset/worsening with exertion | al esfuerzo, al andar, subiendo escaleras, al hacer fuerza, con el ejercicio
  ml_tor_reposo        | Pain at rest / prolonged | en reposo, estando quieto, sin hacer nada, lleva X minutos sin parar, no se le pasa

=== AXIS 2 — DISNEA (qualifiers of ml_dem_disnea) ===
  ml_dis_trabajo        | Increased work of breathing, incl. cannot finish sentences (see rule 16) | tiraje, no puede acabar las frases, respira con dificultad, se le hunden las costillas, jadea, respiración agitada, no puede ni hablar de lo que le cuesta respirar
  ml_dis_ortopnea       | Orthopnea / PND (heart failure) | no puede tumbarse, se ahoga acostado, se despierta ahogándose por la noche, tiene que dormir sentado, incorporado para respirar
  ml_dis_sibilancias    | Wheeze / bronchospasm | pitos al respirar, silba el pecho, sibilancias, le pita el pecho
  ml_dis_cianosis       | Cyanosis | labios morados, se está poniendo morado, azulado, color azul, amoratado
  ml_dis_edemas         | Peripheral edema (right HF) | piernas hinchadas, edemas, hinchazón en los pies, retención de líquidos
  ml_dis_expector       | Productive cough / sputum WITHOUT blood | tos con flemas, expectoración, esputo, echa flemas, tose y saca (mention color if stated)
  ml_dis_hemoptisis     | Blood-streaked sputum / hemoptysis (higher weight: PE, malignancy, TB) | echa sangre por la boca al toser, esputo con sangre, expulsa sangre al toser, hemoptisis
  ml_dis_cuerpo_extrano | Acute choking / suspected FB airway obstruction (finding; the demand is ml_dem_obstr_via_aerea) | se ha atragantado, se atraganta con la comida, algo se le ha ido por otro lado, no puede toser, se le ha quedado algo en la garganta
  ml_dis_tep_riesgo     | VTE/PE risk context (immobilization/surgery/travel; cancer -> ml_ant_oncologico) | operado hace poco, lleva días en la cama, viaje largo, la pierna se le hinchó hace poco, reposo por escayola

=== AXIS 2 — INCONSCIENTE / NIVEL DE CONCIENCIA ===
  ml_inc_no_respira   | # ⚠ REVISAR CRÍTICO — Not breathing / agonal breathing -> possible arrest | no respira, no coge aire, respira raro/agónico, boquea, no le noto respiración, ha dejado de respirar, respiración entrecortada que cesa
  ml_inc_no_responde  | Unresponsive, sustained | no responde, no reacciona, no se despierta, no abre los ojos, no reacciona a nada, inconsciente
  ml_inc_transitorio  | Transient loss recovered (syncope) | se desmayó y ya está, perdió el conocimiento un momento, se recuperó, volvió en sí, lipotimia, desmayado, desplomado
  ml_inc_convul       | Seizure / postictal (finding; the demand is ml_dem_convulsion) | convulsión, convulsiona, ataque epiléptico, espuma por la boca, se muerde la lengua, sacudidas con pérdida de conocimiento
  ml_inc_hipoglucemia | # ⚠ REVISAR — Hypoglycemia SUSPECTED from context, no figure (see ml_vit_hipoglucemia for numeric) | bajada de azúcar, azúcar baja, es diabético y está raro, hipoglucemia, se ha quedado con el azúcar bajo
  ml_inc_pulso_no     | Informational only, does NOT by itself trigger arrest protocol | no le noto el pulso, no le encuentro el pulso, sin pulso, no le late
  ml_confu_aguda      | Acute confusion (responsive but disoriented) | desorientado, dice cosas sin sentido, incoherencias, no sabe dónde está, desvariando, alucinaciones
  ml_trauma_previo    | Trauma/fall preceding the loss of consciousness | se ha caído y se ha dado un golpe, se cayó de la escalera, golpe en la cabeza, accidente antes de encontrarlo así

=== AXIS 2 — HEMORRAGIA (qualifiers of ml_dem_hemorr_* / heridas; NEW in v3) ===
  ml_hem_cuantiosa   | Heavy/uncontrolled bleeding | sangra mucho, a chorros, no para de sangrar, hemorragia abundante, mucha sangre, no cede el sangrado, empapa
  ml_hem_melenas     | Black tarry stools (upper GI) | heces negras, melenas, caca negra, deposiciones negras como alquitrán
  ml_hem_rectorragia | Fresh red rectal blood | sangre roja por el ano, sangra al defecar, rectorragia, sangre en el papel, sangre viva por detrás
  ml_hem_hematemesis | Vomiting blood | vómito con sangre, hematemesis, vomita sangre, posos de café, devuelve sangre
  ml_hem_epistaxis   | Nosebleed | sangra por la nariz, epistaxis, hemorragia nasal, no para de sangrar la nariz
  ml_hem_hematuria   | Blood in urine | sangre en la orina, hematuria, orina con sangre, pis con sangre, orina roja
  ml_hem_vaginal     | Vaginal bleeding (non-obstetric context) | sangrado vaginal, sangra por la vagina, metrorragia, regla muy abundante, pérdidas de sangre

=== AXIS 2 — TRAUMA / MECANISMO (NEW in v3; qualifiers of trauma demands) ===
  ml_trau_alta_energia   | High-energy mechanism | gran velocidad, caída de altura, atrapado, vuelco, salió despedido, aplastado por peso, colisión frontal fuerte
  ml_trau_tce            | Head trauma | golpe en la cabeza, traumatismo craneal, TCE, se ha abierto la cabeza, golpe fuerte en la cabeza
  ml_trau_amputacion     | Amputation / limb crush | amputación, le ha cortado el dedo/mano/pierna, miembro seccionado, aplastamiento de extremidad
  ml_trau_hemorragia_ext | External hemorrhage from traumatic wound | herida que sangra mucho, hemorragia por la herida, sangra por el corte, herida abierta sangrante

=== AXIS 2 — QUEMADURA (NEW in v3; qualifiers of ml_dem_quemaduras) ===
  ml_qem_via_aerea       | # ⚠ CRÍTICO — Airway burn / smoke inhalation | ha respirado humo, inhalación de humo, quemadura en la cara/boca, hollín en la boca, atrapado en el incendio, tose negro
  ml_qem_extensa         | Extensive / large-surface burn | quemadura grande, quemado en gran parte del cuerpo, quemaduras extensas, mucha superficie quemada
  ml_qem_electrica_quimica | Electrical or chemical burn | quemadura eléctrica, quemadura por la corriente, quemadura química, con ácido, con lejía/producto

=== AXIS 2 — OBSTÉTRICO (NEW in v3) ===
  ml_obs_parto_inminente   | # ⚠ CRÍTICO — Imminent delivery | se ve la cabeza, está coronando, empuja/ganas de empujar, contracciones cada pocos minutos, va a nacer ya, no da tiempo a llegar
  ml_obs_sangrado_gestante | Bleeding in a pregnant patient | embarazada sangrando, sangra estando embarazada, pérdidas en el embarazo, hemorragia en gestante

=== AXIS 2 — CONDUCTA SUICIDA (NEW in v3; qualifiers of ml_dem_suicida) ===
  ml_sui_gesto_actual | # ⚠ CRÍTICO — Suicidal act in progress / just done | se ha cortado las venas, se ha tomado las pastillas para quitarse la vida, intento de suicidio, se ha ahorcado, se ha tirado, sobredosis autolítica
  ml_sui_ideacion     | Suicidal ideation without a consummated act | dice que se quiere matar, ideas de suicidio, amenaza con quitarse la vida, no quiere vivir, quiere acabar con todo

=== AXIS 2 — HALLAZGOS COMPARTIDOS (neutral; weight assigned downstream) ===
  ml_veg_sudor      | Cold sweat / pallor / poor color | sudor frío, sudoración, pálido, mal color, blanco, empapado en sudor
  ml_veg_digest     | Nausea / vomiting | náuseas, vómito, vomitando, ganas de vomitar, devolviendo
  ml_veg_mareo      | Dizziness / near-faint (WITHOUT sustained loss — see rule 7) | mareo, vértigo, mareado, sensación de desmayo, aturdido
  ml_dolor_cabeza   | Headache, any intensity (see ml_cefalea_max for thunderclap; both may co-activate) | dolor de cabeza, cefalea, jaqueca, mucho dolor de cabeza, presión en la cabeza
  ml_temblor        | Non-convulsive tremor / rigidity | temblores, temblando, rigidez, sacudidas, tiembla, engarrotado
  ml_disfagia       | Chronic/progressive swallowing difficulty (NOT acute choking) | no traga, dificultad para tragar, no puede tragar, babea al beber
  ml_fiebre         | Fever finding, with or without a figure | tiene fiebre, está ardiendo, se le nota muy caliente, el termómetro marca X, febrícula
  ml_palpitaciones  | Perceived palpitations (TRANSVERSAL — see rule 24) | el corazón le va muy rápido, se le sale del pecho, palpitaciones, nota el corazón acelerado, el corazón le da golpes
  ml_dis_anafilaxia | Anaphylaxis signs (TRANSVERSAL, does NOT require ml_dem_disnea — see rule 24) | se le ha hinchado la cara/labios/lengua, le han salido ronchas, picadura de abeja/avispa con reacción, es alérgico y se lo ha comido, habones por todo el cuerpo, ojos hinchados
  ml_cuadro_catarral| Catarrhal/cold context — PURELY DESCRIPTIVE (see rule 19: never a de-escalation signal) | catarro, resfriado, resfriada, constipado, cuadro catarral, está acatarrada

=== AXIS 2 — TRANSVERSAL: TEMPORAL ===
  ml_tmp_hiperag      | Sudden / very recent onset | ahora, de pronto, de repente, hace minutos, acaba de, en este momento, hace media hora, brusco, repentino
  ml_tmp_despert      | Found on waking / onset unknown | al levantarse, se lo han encontrado así, cuando se despertó, al despertar, se ha levantado con
  ml_tmp_evolucionado | Subacute >12-24h | desde ayer, lleva dos días, desde anoche, lleva varios días, hace tiempo, ayer empezó, la semana pasada

=== AXIS 2 — TRANSVERSAL: FRAGILIDAD ===
  ml_frag_social | Telecare device activation | teleasistencia, televida, securitas, botón de alarma, protección senior
  ml_frag_basal  | Chronic dependency / cognitive impairment | Alzheimer, demencia, silla de ruedas, encamado, deterioro cognitivo, dependiente, postrado, parkinson, vida cama-sillón

=== AXIS 2 — TRANSVERSAL: ANTECEDENTES ===
  ml_ant_cardio     | # ⚠ REVISAR — Prior coronary/HF disease | ya tuvo un infarto, cardiopatía, insuficiencia cardíaca, lleva stent, operado del corazón, angina previa, IAM previo
  ml_ant_acv_prev   | Prior documented stroke | ictus previo, ya le dio un ictus, secuelas de ACV, antecedentes de ictus, le ha repetido
  ml_ant_epoc_asma  | # ⚠ REVISAR — Chronic lung disease (oxygen triggers -> ml_oxigenoterapia) | EPOC, asmático, bronquitis crónica, enfisema, broncópata
  ml_riesgo_cv      | Cardiovascular risk factors — DIAGNOSTIC TERMS ONLY (see rule 18) | hipertensión, HTA, diabetes, diabético, fibrilación auricular, FA, arritmia, colesterol, anticoagulado, marcapasos
  ml_ant_epilepsia  | Known epilepsy | es epiléptico, tiene epilepsia, toma pastillas para las convulsiones, ya le habían dado ataques antes
  ml_ant_tep_prev   | Prior VTE/PE (NEVER folded into ml_ant_cardio — rule 23) | ya tuvo un trombo, tuvo una trombosis, le dio una embolia pulmonar, anticoagulado por un trombo
  ml_ant_oncologico | Active oncologic history (NEVER folded into cardio/epoc — rule 22) | tiene cáncer, con quimioterapia, cáncer de [órgano], tratamiento oncológico, metástasis, oncológico
  ml_oxigenoterapia | Home/ongoing supplemental oxygen, cause-agnostic (rule 20) | tiene oxígeno, oxígeno en casa, oxígeno 24 horas, oxígeno domiciliario, tiene puesto el oxígeno

=== AXIS 2 — TRANSVERSAL: TÓXICOS ===
  ml_tox_consumo | Toxic/alcohol/drug consumption as a FINDING, transversal (the demand is ml_dem_toxicos) | ha bebido, está borracho, se ha tomado pastillas, cocaína, drogas, sobredosis, consumo de tóxicos

=== AXIS 2 — TRANSVERSAL: CONSTANTES OBJETIVAS (⚠ REVISAR thresholds) ===
  ml_vit_hipoxia           | Objective SpO2 <92% | satura 85, saturando a 60, satura a 78, oxígeno a 82-80, sat 85%
  ml_vit_hipoglucemia      | Objective glucose <60 mg/dl | azúcar a 45, glucemia 55, glucosa a 58
  ml_vit_hiperglucemia     | Objective glucose >250 mg/dl | azúcar a 260, glucemia 280, azúcar muy alto a 300
  ml_vit_hipertension      | Objective systolic BP >190 mmHg (rule 21 for colloquial notation) | tensión 200, TA 195/100, la tensión a 19
  ml_vit_hipotension       | Objective systolic BP <90 mmHg | tensión 7/5, TA baja 80/50, la tensión a 8
  ml_vit_bradicardia       | Objective HR/pulse <50 bpm | pulsaciones en 45, FC 40, el pulso muy bajo a 42
  ml_vit_taquicardia       | Objective HR/pulse >130 bpm | pulsaciones 150, FC 140, el corazón muy acelerado a 135
  ml_vit_alterada_referida | Any of the above reported as qualitatively abnormal WITHOUT an exact figure | satura bajo, satura mal, oxígeno bajito, tensión muy alta/baja, el pulso muy alto, el azúcar por los suelos

=== AXIS 2 — TRANSVERSAL: LOGÍSTICA ===
  ml_src_profes  | Caller is a healthcare professional/service | médico del 061, llama enfermera, aviso médico, DCCU, residencia de mayores, auxiliar de ayuda a domicilio, llama médico de guardia
  ml_missing_dat | No useful clinical info (MUTUALLY EXCLUSIVE with every other label) | sin descripción, sin datos, idem, en blanco, no sabe datos

STRICT RULES:
1. Evaluate ONLY the literal text. Do not infer beyond what is written.
2. AXIS 1 is MULTI-LABEL: activate every specific demand described (e.g. chest pain AND dyspnea -> ml_dem_toracico + ml_dem_disnea). Do not force a single demand. The operator ground-truth (IDTIPODEMANDA1) is single-label, so multiple demand labels are EXPECTED and legitimate.
3. ml_missing_dat is MUTUALLY EXCLUSIVE with all other labels. Activate ONLY when the field is truly empty / equivalent to "no data". Objective vitals (ml_vit_*), a fever mention, an oncologic history, a bare demand, etc. all count as data.
4. ml_foc_lateral activates ONLY if a body side is explicitly mentioned.
5. Demand labels may activate on suspicion ("posible ictus", "parece un infarto", "puede ser un cólico").
6. ml_foc_habla vs ml_confu_aguda: cannot produce/articulate speech -> ml_foc_habla; speaks but incoherent -> ml_confu_aguda. Both may activate.
7. CONSCIOUSNESS PRIORITY: sustained unresponsiveness -> ml_inc_no_responde (and NOT ml_veg_mareo). Brief recovered faint -> ml_inc_transitorio. If both dizziness and sustained unresponsiveness are described, unresponsiveness takes priority.
8. # ⚠ REVISAR CRÍTICO: ml_inc_no_respira is the highest-priority safety label. Activate on ANY explicit indication of absent or agonal breathing. When in doubt between ml_inc_no_respira and ml_inc_no_responde, activate BOTH. If the text describes both absent breathing AND unresponsiveness as the reason for the call, also activate ml_dem_pcr.
9. ml_foc_visual: only for a conscious patient with explicit visual/eye-movement deficit. "No abre los ojos", "mirada perdida" -> consciousness labels, NOT ml_foc_visual. A pure eye complaint (foreign body, red eye, ocular pain) with no neuro context -> ml_dem_oftalmologico, NOT ml_foc_visual.
10. ml_inc_convul vs ml_temblor: convulsive features (loss of control, tongue biting, postictal) -> ml_inc_convul (+ ml_dem_convulsion if that is the demand); isolated tremor/rigidity -> ml_temblor.
11. ml_tmp_evolucionado vs ml_tmp_hiperag: MUTUALLY EXCLUSIVE. If remote onset AND acute worsening are both described, activate ml_tmp_evolucionado only.
12. # ⚠ REVISAR: ml_veg_sudor / ml_veg_digest / ml_veg_mareo are NEUTRAL. Activate whenever described, regardless of demand. Do NOT suppress them as stroke mimics; weight is decided downstream.
13. ml_tor_pleuritico and ml_tor_opresivo may both activate if both qualities are described.
14. Antecedent labels (ml_ant_*, ml_ant_epilepsia, ml_ant_tep_prev, ml_ant_oncologico) describe PRIOR history, not the current episode. A current infarction is a demand, not ml_ant_cardio.
15. Return ONLY a valid JSON array of strings (the active label names). No explanation, no markdown, no text outside the JSON array.
16. ml_foc_habla vs ml_dis_trabajo: if inability to speak is caused by breathlessness, activate ml_dis_trabajo, NOT ml_foc_habla. Reserve ml_foc_habla for genuine speech/articulation impairment. Both may co-activate only if a real speech alteration is described in addition to breathlessness.
17. ml_dem_inconsciente vs specific consciousness demands: "desmayado/desplomado/inconsciente/no responde" activate ml_dem_inconsciente UNLESS the text frames a different demand (ictus/torácico/disnea/convulsión) as the reason — then activate only the specific finding (ml_inc_*) with that other demand. A brief recovered faint framed as the reason -> ml_dem_mareo_sincope (+ ml_inc_transitorio), NOT ml_dem_inconsciente.
18. Numeric vital vs risk-factor antecedent: a bare numeric BP/HR/glucose/SpO2 reading activates ONLY the ml_vit_* label (if it crosses the threshold), NEVER ml_riesgo_cv by itself. Diagnostic terms (HTA, diabético, FA, arritmia, colesterol, anticoagulado, marcapasos) activate ml_riesgo_cv regardless of any number. Both may co-activate.
19. ml_cuadro_catarral carries NO implication of lower severity. Activate whenever a catarrhal/cold context is mentioned, even alongside high-severity labels.
20. ml_oxigenoterapia vs ml_ant_epoc_asma: oxygen use alone activates ONLY ml_oxigenoterapia. ml_ant_epoc_asma requires an explicit diagnostic term (EPOC, asmático, bronquitis crónica, enfisema, broncópata). Both may co-activate.
21. ml_vit_* thresholds: hipoxia SpO2<92; hipoglucemia <60; hiperglucemia >250; hipertensión sistólica >190; hipotensión sistólica <90; bradicardia <50; taquicardia >130. Colloquial Spanish BP often drops the trailing zero ("19/9"=190/90, "7/5"=70/50, "15/10"=150/100) — parse before comparing.
22. ml_ant_oncologico is independent; NEVER fold into ml_ant_cardio or ml_ant_epoc_asma.
23. ml_ant_tep_prev is independent; NEVER fold into ml_ant_cardio.
24. ml_palpitaciones and ml_dis_anafilaxia are TRANSVERSAL findings: may activate with any demand or none; do not gate on ml_dem_disnea/ml_dem_toracico.
25. DEMAND vs OBJECTIVE VITAL (glucose/BP): the demand labels ml_dem_hipoglucemia/ml_dem_hiperglucemia/ml_dem_hipertension/ml_dem_hipotension describe that the CALL IS ABOUT that parameter. The ml_vit_* label additionally activates ONLY when an explicit figure crosses the threshold. They co-activate when the call is about it AND a threshold figure is given (e.g. "diabético con el azúcar a 40" -> ml_dem_hipoglucemia + ml_vit_hipoglucemia). A demand may activate without ml_vit_* (no figure), and ml_vit_* may activate under another demand (an incidental reading).
26. HEMORRHAGE FAMILY: pick the demand by SOURCE — anal/black stools -> ml_dem_hemorr_anal; mouth/nose/vomited blood -> ml_dem_hemorr_oral; urine -> ml_dem_hemorr_urinaria; vaginal (non-pregnant) -> ml_dem_hemorr_gineco; a bleeding traumatic/unspecified wound -> ml_dem_heridas_hemorr; stab/gunshot -> ml_dem_arma_blanca_fuego. If the source is unclear, use ml_dem_heridas_hemorr. Always add the matching ml_hem_* finding. A pregnant patient who bleeds -> ml_dem_amenaza_aborto (+ ml_obs_sangrado_gestante), NOT ml_dem_hemorr_gineco.
27. DOLOR FAMILY: pick the demand by LOCATION — chest -> ml_dem_toracico; abdomen -> ml_dem_dolor_abdo; neck -> ml_dem_dolor_cervical; back/spine -> ml_dem_dolor_espalda; limb/joint -> ml_dem_dolor_extrem; genitourinary/pelvic -> ml_dem_dolor_genitourin; head -> ml_dem_cefalea; diffuse/ill-defined -> ml_dem_dolor_generaliz. Multiple locations -> multiple demands.
28. TRAUMA MECHANISM: a fall -> ml_dem_caida; fall from height/jump -> ml_dem_precipitacion; blunt blow -> ml_dem_contusion; traffic -> ml_dem_accid_transp; crush/amputation -> ml_dem_aplast_amput (+ ml_trau_amputacion). Add ml_trau_* qualifiers (alta_energia, tce, amputacion, hemorragia_ext) when described. A head blow that precedes loss of consciousness also activates ml_trauma_previo.
29. SUICIDE vs INTOXICATION: an intentional self-harm act or overdose with autolytic intent -> ml_dem_suicida + ml_sui_gesto_actual. A non-intentional intoxication/overdose -> ml_dem_toxicos + ml_tox_consumo. Ideation without an act -> ml_dem_suicida + ml_sui_ideacion.
30. ADMINISTRATIVE / LOW-CONTENT demands (ml_dem_autoactiv, ml_dem_enfermeria, ml_dem_otros_valdificil, ml_dem_cronico_complejo, ml_dem_paliativos): activate only when the text explicitly matches; do NOT use them as a fallback for unclear clinical text. If the text is genuinely empty of clinical info, use ml_missing_dat instead.

OUTPUT FORMAT — return ONLY this, nothing else:
["ml_label1", "ml_label2"]
or [] if nothing applies (should be extremely rare).
"""
USER_TEMPLATE = 'MOTIVOLITERAL: "%s"'

# Structured output: an array of unique taxonomy labels (the output format requested in the prompt)
OUTPUT_SCHEMA = {'type': 'array', 'uniqueItems': True,
                 'items': {'type': 'string', 'enum': ML_COLUMNS}}

# Every taxonomy label must be described in the prompt
_prompt_labels = set(re.findall(r'\bml_\w+', SYSTEM_PROMPT))
_missing = [c for c in ML_COLUMNS if c not in _prompt_labels]
assert not _missing, f'Labels not described in the prompt: {_missing}'

## 4. Normalisation and LLM call

Texts are normalised (accents removed, lower-case, whitespace collapsed; punctuation kept because it separates findings). Empty-equivalent tokens are not sent to the model. Each distinct normalised text is classified once, one text per request, with the output constrained by the JSON schema. Labels outside the taxonomy are discarded (a safeguard; the schema already prevents them). A request that still fails after the retries leaves the text unclassified (`None`).

Ollama truncates prompts longer than `num_ctx` without raising an error, which would silently remove part of the taxonomy. A probe request on a synthetic text, sent to a freshly loaded model, therefore checks that the full prompt fits in the context window and that the output is a JSON array.

*Note on determinism:* with `temperature = 0` repeated runs are expected to be highly consistent but are not guaranteed to be bitwise identical.

In [ ]:
EMPTY_EQUIVALENT = {'', 'nan', 'none', 'no', 'ninguno', 'null', 'idem', 'id', '-', '.'}

def normalize_literal(text):
    s = str(text).strip().lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', s).strip()

def _chat(text, model=MODEL):
    return ollama.chat(model=model, format=OUTPUT_SCHEMA, options=LLM_OPTIONS, keep_alive=KEEP_ALIVE,
                       messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                                 {'role': 'user', 'content': USER_TEMPLATE % text}])

def classify(text, model=MODEL):
    """Raw (pre-remediation) label list for one normalised text, or None if the request fails."""
    for _ in range(RETRIES + 1):
        try:
            resp = _chat(text, model)
        except Exception:  # transport error: retry
            continue
        try:
            labels = json.loads(resp['message']['content'])
        except (ValueError, KeyError, TypeError):  # malformed output: not retried
            return None
        if not isinstance(labels, list):
            return None
        labels = dict.fromkeys(x for x in labels if isinstance(x, str))
        return [c for c in labels if c in ML_COLUMNS_SET]
    return None

demands['TEXT_NORM'] = demands['text'].fillna('').map(normalize_literal)
demands['HAS_TEXT'] = ~demands['TEXT_NORM'].isin(EMPTY_EQUIVALENT)
print(f'Demands with usable text: {demands["HAS_TEXT"].sum():,}')

In [ ]:
# Prompt-fit probe on a synthetic text. The model is unloaded first so that no cached prefix
# lowers the reported prompt token count.
ollama.generate(model=MODEL, prompt='', keep_alive=0)
_probe = _chat('dolor toracico opresivo con sudoracion')
n_prompt = _probe['prompt_eval_count']
print(f'Prompt tokens: {n_prompt:,} | num_ctx: {LLM_OPTIONS["num_ctx"]:,} | output: {_probe["message"]["content"]}')
assert n_prompt + LLM_OPTIONS['num_predict'] <= LLM_OPTIONS['num_ctx'], 'prompt would be truncated'
assert isinstance(json.loads(_probe['message']['content']), list), 'output is not a JSON array'

## 5. Model selection (optional)

With `RUN_BENCHMARK = True`, each candidate model classifies the same random sample of unique texts. We report seconds per text (after one warm-up request, so model loading is excluded), failed requests and mean Jaccard agreement with a reference model. Agreement is not accuracy. <Describe the clinical review of discrepancies used to select `MODEL`.>

In [ ]:
RUN_BENCHMARK = False
CANDIDATE_MODELS = ['<candidate model 1>', '<candidate model 2>', 'qwen2.5:32b-instruct-q3_K_M']  # last = reference
N_BENCH = 60

def jaccard(a, b):
    u = a | b
    return 1.0 if not u else len(a & b) / len(u)

if RUN_BENCHMARK:
    pool = sorted(demands.loc[demands['HAS_TEXT'], 'TEXT_NORM'].unique())
    sample = random.Random(SEED).sample(pool, min(N_BENCH, len(pool)))
    outputs, rows = {}, []
    for m in CANDIDATE_MODELS:
        classify(sample[0], model=m)  # warm-up
        t0 = time.perf_counter()
        outputs[m] = [classify(t, model=m) for t in sample]
        dt = time.perf_counter() - t0
        rows.append({'model': m, 'n': len(sample), 'sec_per_text': round(dt / len(sample), 2),
                     'failed': sum(o is None for o in outputs[m])})
    ref = CANDIDATE_MODELS[-1]
    for r in rows:
        r['jaccard_vs_reference'] = round(float(np.mean(
            [jaccard(set(a or []), set(b or [])) for a, b in zip(outputs[r['model']], outputs[ref])])), 3)
    print(pd.DataFrame(rows).to_string(index=False))

## 6. LLM classification

Unique texts are processed in descending order of frequency, so that a partial run covers the largest number of demands. Results are cached (one JSON line per text) in a file whose name is a hash of the model, options, prompt and schema, so the run can be resumed and a configuration change never reuses stale labels. Texts whose request failed are not cached and are retried on resumption.

In [ ]:
_config = json.dumps({'model': MODEL, 'options': LLM_OPTIONS, 'system': SYSTEM_PROMPT,
                      'user': USER_TEMPLATE, 'schema': OUTPUT_SCHEMA}, sort_keys=True)
LLM_CACHE = OUT_DIR / f'llm_cache_reason_{hashlib.sha256(_config.encode()).hexdigest()[:12]}.jsonl'

unique_texts = list(demands.loc[demands['HAS_TEXT'], 'TEXT_NORM'].value_counts().index)  # most frequent first

cache = {}
if LLM_CACHE.exists():
    with open(LLM_CACHE, encoding='utf-8') as f:
        for line in f:
            try:
                d = json.loads(line)
            except ValueError:  # incomplete last line after an interruption
                continue
            cache[d['text']] = d['labels']
pending = [t for t in unique_texts if t not in cache]
print(f'Unique texts: {len(unique_texts):,} | cached: {len(cache):,} | pending: {len(pending):,}')

with open(LLM_CACHE, 'a', encoding='utf-8') as f, ThreadPoolExecutor(N_WORKERS) as ex:
    for start in range(0, len(pending), CHUNK):
        batch = pending[start:start + CHUNK]
        for text, labels in zip(batch, ex.map(classify, batch)):
            if labels is not None:
                cache[text] = labels
                f.write(json.dumps({'text': text, 'labels': labels}, ensure_ascii=False) + '\n')
        f.flush()
        print(f'  processed {min(start + CHUNK, len(pending)):,} / {len(pending):,}')

print(f'Unresolved texts: {sum(t not in cache for t in unique_texts):,}')

## 7. Deterministic remediation

The pipeline supports deterministic, traceable rules applied to the normalised text and the LLM labels (same design as in the past-history notebook). Each rule may only add or remove taxonomy labels and records a tag in `TRACE`. <No rules were applied in the reported run. / List each rule with the systematic error that motivated it.>

In [ ]:
# Each rule: f(text_norm, labels: set) -> (labels: set, tag or None)
REMEDIATION_RULES = []

def remediate(text_norm, labels):
    """Returns (remediated labels in taxonomy order, trace)."""
    L, tags = set(labels), []
    for rule in REMEDIATION_RULES:
        L, tag = rule(text_norm, L)
        if tag:
            tags.append(tag)
    return [c for c in ML_COLUMNS if c in L], '; '.join(tags)

remediated = {t: remediate(t, labs) for t, labs in cache.items()}
_changed = sum(set(v[0]) != set(cache[t]) for t, v in remediated.items())
print(f'Unique texts whose labels changed after remediation: {_changed:,} of {len(remediated):,}')
print('Traced rules:', dict(Counter(r for _, tr in remediated.values() if tr for r in tr.split('; '))))

## 8. Consolidation per demand

Labels are attached to each demand through its normalised text and split by axis. For each axis, the readable column lists the labels or, if there are none, the availability label (`SIN_HALLAZGO`, `SIN_TEXTO` or `NO_CLASIFICADO`).

In [ ]:
_pairs = list(zip(demands['TEXT_NORM'], demands['HAS_TEXT']))
_lab = [remediated[t][0] if (h and t in remediated) else None for t, h in _pairs]
demands['TRACE'] = [remediated[t][1] if (h and t in remediated) else '' for t, h in _pairs]
demands['estado'] = ['SIN_TEXTO' if not h else 'NO_CLASIFICADO' if L is None else 'OK' if L else 'SIN_HALLAZGO'
                     for (_, h), L in zip(_pairs, _lab)]
demands['labels'] = [L or [] for L in _lab]
demands['labels_dem'] = [[c for c in L if c.startswith('ml_dem_')] for L in demands['labels']]
demands['labels_hall'] = [[c for c in L if not c.startswith('ml_dem_')] for L in demands['labels']]
demands['n_dem'] = demands['labels_dem'].str.len()
demands['n_hall'] = demands['labels_hall'].str.len()
demands['tiene_texto'] = demands['HAS_TEXT'].astype(int)

def axis_value(labels, status):
    if labels:
        return ', '.join(labels)
    return 'SIN_HALLAZGO' if status in ('OK', 'SIN_HALLAZGO') else status

demands['EJE1_DEMANDA'] = [axis_value(L, s) for L, s in zip(demands['labels_dem'], demands['estado'])]
demands['EJE2_HALLAZGOS'] = [axis_value(L, s) for L, s in zip(demands['labels_hall'], demands['estado'])]

_cls = demands['estado'].isin(['OK', 'SIN_HALLAZGO'])
print(f'Demands: {len(demands):,}')
print(demands['estado'].value_counts().to_string())
print(f'With >=1 Axis-1 label: {(demands["n_dem"] > 0).sum():,} | with >=1 Axis-2 label: {(demands["n_hall"] > 0).sum():,}')
print(f'Mean labels per classified demand: Axis 1 {demands.loc[_cls, "n_dem"].mean():.2f} | '
      f'Axis 2 {demands.loc[_cls, "n_hall"].mean():.2f}')

## 9. One-hot encoding (one block per axis)

Columns cover the full vocabulary (including labels with zero prevalence) in a stable order: `dem_*` for Axis 1 and `hall_*` for Axis 2. Availability is carried by `tiene_texto` and `estado`; demands with `NO_CLASIFICADO` have all indicators at 0 and should be excluded from analyses.

In [ ]:
def one_hot(lists, vocab, prefix):
    pos = {c: j for j, c in enumerate(vocab)}
    M = np.zeros((len(lists), len(vocab)), dtype=np.int8)
    rows = [i for i, L in enumerate(lists) for c in L if c in pos]
    cols = [pos[c] for L in lists for c in L if c in pos]
    M[rows, cols] = 1
    return pd.DataFrame(M, columns=[prefix + c for c in vocab])

ohe_dem = one_hot(demands['labels_dem'].tolist(), DEM_COLS, 'dem_')
ohe_hall = one_hot(demands['labels_hall'].tolist(), FIND_COLS, 'hall_')

BASE_COLS = [KEY_DEM, 'EJE1_DEMANDA', 'EJE2_HALLAZGOS', 'n_dem', 'n_hall', 'tiene_texto', 'estado']
features = pd.concat([demands[BASE_COLS].reset_index(drop=True), ohe_dem, ohe_hall], axis=1)
print(f'{ohe_dem.shape[1]} Axis-1 and {ohe_hall.shape[1]} Axis-2 indicator columns over {len(features):,} demands')

## 10. Diagnostics without a reference standard

These do not measure accuracy. (a) **Lexical flags**: texts containing an unambiguous trigger term whose expected label is absent are listed as candidate false negatives for manual review; a negation within the two words preceding the trigger suppresses the flag (except for the not-breathing flag, whose trigger is itself a negation). (b) **Rule violations**: co-occurrences forbidden by the prompt's mutual-exclusivity rules (3 and 11) are counted, not corrected. <Report the manual review and/or the evaluation against an annotated reference sample used in the paper.>

In [ ]:
NEG_BEFORE_RE = re.compile(r'\b(no|niega|niegan|sin|descart\w*|nunca|ausencia de)\W+(\w+\W+){0,2}$')
FLAGS = [  # (trigger, expected label, flag name, negation guard)
    (re.compile(r'\bictus\b|\bacva?\b|\bait\b|derrame cerebral'), 'ml_dem_ictus', 'stroke_missed', True),
    (re.compile(r'dolor.{0,15}pecho|dolor toracic|opresion.{0,10}pecho'), 'ml_dem_toracico', 'chest_pain_missed', True),
    (re.compile(r'no respira|no coge aire|parada card'), 'ml_inc_no_respira', 'not_breathing_missed', False),
    (re.compile(r'convuls'), 'ml_dem_convulsion', 'seizure_missed', True),
    (re.compile(r'se ha caido|se cayo|\bcaida'), 'ml_dem_caida', 'fall_missed', True),
]

def flag_text(t, labels):
    out = []
    for rgx, label, name, neg_guard in FLAGS:
        m = rgx.search(t)
        if m and label not in labels and not (neg_guard and NEG_BEFORE_RE.search(t[:m.start()])):
            out.append(name)
    return out

text_freq = demands.loc[demands['HAS_TEXT'], 'TEXT_NORM'].value_counts()
flags_texts, flags_demands, review = Counter(), Counter(), []
for t, n in text_freq.items():
    if t in remediated:
        for name in flag_text(t, remediated[t][0]):
            flags_texts[name] += 1
            flags_demands[name] += n
            review.append({'flag': name, 'text': t, 'labels': remediated[t][0], 'n_demands': n})
review_df = pd.DataFrame(review)  # kept in memory for manual review; not saved (contains free text)

print('Lexical flags (candidates for review): unique texts | demands')
for _, _, name, _ in FLAGS:
    print(f'  {name:<22} {flags_texts[name]:>8,} | {flags_demands[name]:>9,}')

EXCLUSIVITY = {
    'ml_missing_dat with other labels (rule 3)': lambda L: 'ml_missing_dat' in L and len(L) > 1,
    'ml_tmp_hiperag with ml_tmp_evolucionado (rule 11)':
        lambda L: 'ml_tmp_hiperag' in L and 'ml_tmp_evolucionado' in L,
}
print('Rule violations (demands):')
for name, test in EXCLUSIVITY.items():
    print(f'  {name}: {sum(test(L) for L in demands["labels"]):,}')

prev = Counter(c for L in demands['labels'] for c in L)
print('Most prevalent labels (% of demands):')
for c, n in prev.most_common(15):
    print(f'  [{"DEM" if c.startswith("ml_dem_") else "FIND"}] {c:<32} {n:>10,} ({n / len(demands):.1%})')

## 11. Integrity checks

These verify structural consistency, not clinical accuracy.

In [ ]:
status = features['estado']
dcols, hcols = list(ohe_dem.columns), list(ohe_hall.columns)
n_lab = features['n_dem'] + features['n_hall']
used = set(c for L in demands['labels'] for c in L)
checks = {
    'demand_id unique': features[KEY_DEM].is_unique,
    'all cohort demands preserved': len(features) == cohort[KEY_DEM].dropna().nunique(),
    'status partition is exhaustive': status.isin(['OK', 'SIN_HALLAZGO', 'SIN_TEXTO', 'NO_CLASIFICADO']).all(),
    'labels within taxonomy': used <= ML_COLUMNS_SET,
    'axis dimensions (61 / 85)': len(dcols) == 61 and len(hcols) == 85,
    'one-hot row sums equal n_dem': (features[dcols].sum(axis=1) == features['n_dem']).all(),
    'one-hot row sums equal n_hall': (features[hcols].sum(axis=1) == features['n_hall']).all(),
    'OK implies at least one label': (n_lab[status.eq('OK')] > 0).all(),
    'non-OK implies no labels': (n_lab[status.ne('OK')] == 0).all(),
    'SIN_TEXTO iff no text': (status.eq('SIN_TEXTO') == features['tiene_texto'].eq(0)).all(),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
print(f'Unclassified demands (NO_CLASIFICADO): {status.eq("NO_CLASIFICADO").sum():,}')
assert all(checks.values())

## 12. Output

In [ ]:
demands[BASE_COLS + ['TRACE']].to_parquet(OUT_DIR / 'reason_per_demand.parquet', index=False)
features.to_parquet(OUT_DIR / 'reason_one_hot.parquet', index=False)